## Imports

In [1]:
import os
import re
import time

import numpy as np
import pandas as pd
import hiplot as hip
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'

In [3]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

## Datasets & Parameter

In [4]:
## Parameter 
start_date = '20240318'
end_date = '20240331'

In [5]:
# Get the current working directory
cwd = os.getcwd()
print(cwd)
local_datasets = '/Users/rapido/local-datasets/customer/appography/appography_v1/city/hyderabad/'
print(local_datasets)

/Users/rapido/analytics/customer/Appography/Appography_v1
/Users/rapido/local-datasets/customer/appography/appography_v1/city/hyderabad/


In [6]:
##usecase_tag

usecase_tag_query = f"""

WITH active_customers AS (

    SELECT 
        customer_id,
        order_id,
        drop_location_hex_10 hex_10
    FROM 
        orders.order_logs_snapshot
    WHERE
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        AND channel_host = 'android'
        AND city_name = 'Hyderabad'
    GROUP BY 1,2,3
    ), 

    use_case AS (
    
    SELECT
        hex_10, 
        usecase_tag
    FROM
        (
        SELECT 
            hex_10, 
            combined_final_usecase_accuracy as usecase_tag,
            ROW_NUMBER() OVER(PARTITION BY hex_10 ORDER BY run_date DESC) seq_no
        FROM
            hive.experiments_internal.combined_usecase_hex_10_level
        WHERE 
            aoi = 'Hyderabad, India'
        )
    WHERE   
        seq_no = 1
    ),
    
    merge AS (
    SELECT
        customer_id,
        usecase_tag,
        ROW_NUMBER() OVER(PARTITION BY customer_id ORDER BY orders DESC) seq_no
    FROM 
        (
        SELECT
            a.customer_id,
            COALESCE(b.usecase_tag, 'Unknown') usecase_tag,
            COUNT(DISTINCT order_id) orders
        FROM 
            active_customers a
        LEFT JOIN 
            use_case b
            ON a.hex_10 = b.hex_10
        GROUP BY 1,2
        )
    WHERE 
        usecase_tag != 'Unknown'
    )
    
    SELECT
        a.customer_id,
        COALESCE(b.usecase_tag, 'Unknown') usecase_tag
    FROM 
        active_customers a
    LEFT JOIN 
        merge b 
        ON a.customer_id = b.customer_id
        AND seq_no = 1
    GROUP BY 1,2

"""

df_usecase_tag_query = pd.read_sql(usecase_tag_query, connection)
df_usecase_tag_query.to_csv(local_datasets + 'raw/usecase_tag_v1.csv', index=False)

In [7]:
df_usecase_tag = pd.read_csv(local_datasets + 'raw/usecase_tag_v1.csv')
print(df_usecase_tag.shape)
df_usecase_tag.head(2)

(1770811, 2)


,customer_id,usecase_tag
0,61c5dbead82740e8b41dc0d4,residential
1,63c6de55926f778ec1471981,residential


### Active customer (RR-customers)

In [8]:
df_bangalore_active_customer = pd.read_csv(local_datasets + 'raw/hyderabad_customers_v1.csv')
df_bangalore_active_customer = df_bangalore_active_customer.drop('app_list_set', axis=1)
print(df_bangalore_active_customer.shape)
df_bangalore_active_customer.head(2)

(1614552, 16)


,customer_id,gender,rapido_age,customer_age,fe_count,rr_count,net_count,ltr_segment,life_time_rides,life_time_stage,income_segment,service_affinity,distance_preference,price_sensitivity,net_count_with_nan,rpc
0,6370bc7bee51da794a7f7727,MALE,511.0,NaN,12.0,8.0,8.0,PHH,75,COMMITTED,MEDIUM_INCOME,ONLY_LINK,SHORT_DISTANCE,NPS,8.0,d. high rpc
1,5d1aec85668011467e27c146,MALE,1135.0,28.0,5.0,2.0,0.0,PHH,37,INACTIVE,MEDIUM_INCOME,ONLY_LINK,LONG_DISTANCE,NPS,NaN,a. zero rpc


In [9]:
df_bangalore_active_customer = pd.merge(df_bangalore_active_customer, df_usecase_tag,
                                        how='left', on=['customer_id'])

### customer installed apps

In [10]:
df_customer_mapped = pd.read_csv(local_datasets + 'processed/customer_app_categories_mapped_v1.csv')
print(df_customer_mapped.shape)
df_customer_mapped.head(2)

(1613086, 14)


,customer_id,app_list,app_count,categories_l1,categories_l1_count,categories_l2,categories_l2_count,Driver_App,Educational,Gaming,Finance_Investment,Office,Ride haling,Rest
0,5737c6d3ddbec2203f7332ba,"['myntra', 'telegram', 'pharmeasy', 'instagram...",17,"['Tools', 'Social', 'Delivery_Food', 'OTT', 'S...",8,"['Ride haling', 'Office', 'Rest']",3,0,0,0,0,1,1,1
1,573f28f39b0ffc283676efee,"['telegram', 'blusmart', 'instagram', 'cred', ...",47,"['Tools', 'Social', 'Health', 'Finance_News', ...",16,"['Finance_Investment', 'Ride haling', 'Office'...",4,0,0,0,1,1,1,1


### Customer app & cat explode mapping

In [11]:
df_customer_app_cat_mapping = pd.read_csv(local_datasets + 'processed/customer_app_categories_mapping_v1.csv')
df_customer_cat_mapping = df_customer_app_cat_mapping[['customer_id', 
#                                                        'categories_l1', 
                                                       'categories_l2']]\
                            .drop_duplicates()
df_customer_cat_mapping.head(1)

,customer_id,categories_l2
0,6370bc7bee51da794a7f7727,Rest


In [12]:
total_customers = df_customer_cat_mapping.customer_id.nunique()
total_customers

1613086

In [13]:
print(df_customer_app_cat_mapping['categories_l1'].unique())

['Social' 'Ecommerce' 'Streaming_Music' 'Delivery_Food'
 'Finance_Transactions' 'OTT' 'Tools' 'Driver_App' 'News'
 'Travel_Ridehailing' 'Office' 'Gaming' 'Wearable' 'Travel_Bookings'
 'Educational' 'Entertainment' 'Delivery_Grocery' 'Finance_Investment'
 'Health' 'Dating' 'Fantasy_Sports' 'Finance_News' 'Devotional']


In [14]:
print(df_customer_app_cat_mapping['categories_l2'].unique())

['Rest' 'Driver_App' 'Ride haling' 'Office' 'Gaming' 'Educational'
 'Finance_Investment']


In [15]:
# single_category = ['Travel_Ridehailing']

# ### Office
# print(single_category)
# df_temp = df_customer_app_cat_mapping[df_customer_app_cat_mapping['categories_l1'].isin(single_category)]\
#             .groupby(['categories_l1','app_list'])\
#             .agg(customers = ('customer_id','nunique'))\
#             .sort_values(['customers'],ascending=False)\
#             .reset_index()
# df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
# df_temp

In [16]:
### Office
df_temp = df_customer_app_cat_mapping[~df_customer_app_cat_mapping['categories_l2'].isin(['Rest'])]\
            .groupby(['categories_l2','app_list_set'])\
            .agg(customers = ('customer_id','nunique'))\
            .sort_values(by=['categories_l2', 'customers'], ascending=[False, False])\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,categories_l2,app_list_set,customers,%
0,Ride haling,ola,830261,51.47
1,Ride haling,uber,775813,48.09
2,Ride haling,in drive,20133,1.25
3,Ride haling,namma yatri,17050,1.06
4,Ride haling,quick ride,6792,0.42
5,Ride haling,driveu driver app,4399,0.27
6,Ride haling,blusmart,2750,0.17
7,Ride haling,quickride,982,0.06
8,Ride haling,jugnoo,513,0.03
9,Office,zoom,446570,27.68


### Merge raw data

In [17]:
df_customer_data = pd.merge(df_bangalore_active_customer, df_customer_mapped,
                            how='inner', on=['customer_id']
                           )
print(df_customer_data.shape)
df_customer_data.head(1)

(1613086, 30)


,customer_id,gender,rapido_age,customer_age,fe_count,rr_count,net_count,ltr_segment,life_time_rides,life_time_stage,income_segment,service_affinity,distance_preference,price_sensitivity,net_count_with_nan,rpc,usecase_tag,app_list,app_count,categories_l1,categories_l1_count,categories_l2,categories_l2_count,Driver_App,Educational,Gaming,Finance_Investment,Office,Ride haling,Rest
0,6370bc7bee51da794a7f7727,MALE,511.0,NaN,12.0,8.0,8.0,PHH,75,COMMITTED,MEDIUM_INCOME,ONLY_LINK,SHORT_DISTANCE,NPS,8.0,d. high rpc,health_and_personal,"['myntra', 'instagram', 'wynk music', 'twitter...",7,"['Social', 'Delivery_Food', 'OTT', 'Streaming_...",6,['Rest'],1,0,0,0,0,0,0,1


#### Derived columns

In [18]:
## RPC

df_customer_data['rpc'] =  df_customer_data['net_count'].replace(0, np.nan)
df_customer_data.rpc.describe()

count    1.389223e+06
mean     2.778540e+00
std      3.231849e+00
min      1.000000e+00
25%      1.000000e+00
50%      2.000000e+00
75%      3.000000e+00
max      1.530000e+02
Name: rpc, dtype: float64

In [19]:
df_customer_data['rpc_bucket'] = np.where(df_customer_data['net_count'] < 1 , 'a. ZERO',
                                    np.where(df_customer_data['net_count'] < 2 , 'b. LOW',
                                        np.where(df_customer_data['net_count'] < 4 , 'c. MEDIUM', 
                                            np.where(df_customer_data['net_count'] >= 4 , 'd. HIGH', None))))

In [20]:
## app_count_bucket

df_customer_data.app_count.describe()

count    1.613086e+06
mean     1.756618e+01
std      8.856690e+00
min      1.000000e+00
25%      1.100000e+01
50%      1.600000e+01
75%      2.300000e+01
max      9.400000e+01
Name: app_count, dtype: float64

In [21]:
df_customer_data['app_count_bucket'] = np.where(df_customer_data['net_count'] < 5 , '1-5',
                                        np.where(df_customer_data['net_count'] < 10 , '6-10',
                                          np.where(df_customer_data['net_count'] < 16 , '11-15', 'Above 15')))

In [22]:
## category_count_bucket

df_customer_data['category_count'] =  df_customer_data['categories_l2_count'].replace(0, 1)
df_customer_data.category_count.describe()

count    1.613086e+06
mean     2.925554e+00
std      1.235463e+00
min      1.000000e+00
25%      2.000000e+00
50%      3.000000e+00
75%      4.000000e+00
max      7.000000e+00
Name: category_count, dtype: float64

In [23]:
df_customer_data['category_count_bucket'] = np.where(df_customer_data['category_count'] < 2 , '1',
                                              np.where(df_customer_data['category_count'] < 4 , '2-3','Above 3'))

In [24]:
## category_count

# df_customer_data['category_count'] =  df_customer_data['categories_l2_count'].replace(0, 1)
df_customer_data.rapido_age.describe()

count    1.612795e+06
mean     7.548420e+02
std      5.611305e+02
min      1.000000e+00
25%      3.030000e+02
50%      6.610000e+02
75%      1.088000e+03
max      2.876000e+03
Name: rapido_age, dtype: float64

#### Merge

In [25]:
df_customer_data_explode = pd.merge(df_customer_data,
                                    df_customer_cat_mapping,
                                    how='inner', on =['customer_id'])

df_customer_data_explode[df_customer_data_explode['customer_id'] == '6475f56aba9a99b915ccc086'].head()

,customer_id,gender,rapido_age,customer_age,fe_count,rr_count,net_count,ltr_segment,life_time_rides,life_time_stage,income_segment,service_affinity,distance_preference,price_sensitivity,net_count_with_nan,rpc,usecase_tag,app_list,app_count,categories_l1,categories_l1_count,categories_l2_x,categories_l2_count,Driver_App,Educational,Gaming,Finance_Investment,Office,Ride haling,Rest,rpc_bucket,app_count_bucket,category_count,category_count_bucket,categories_l2_y


## User Base Distribution analysis

LTR-Segment

Service Affinity

Distance Affinity

Customers Use Case

AO- Net Funnel

Gender

Age (Whatever fill rate we have)

Income

RPC

In [26]:
tot_customers = df_customer_cat_mapping.customer_id.nunique()
df_category_agg = df_customer_cat_mapping\
                    .groupby(['categories_l2'])\
                    .agg(total_customers = ('customer_id','nunique'))\
                    .sort_values(['total_customers'], ascending=False)\
                    .reset_index()
df_category_agg['% distribution'] =  (df_category_agg['total_customers']*100.00/tot_customers).round(2)
df_category_agg

,categories_l2,total_customers,% distribution
0,Rest,1612189,99.94
1,Ride haling,1074068,66.58
2,Office,867807,53.80
3,Finance_Investment,428267,26.55
4,Gaming,283452,17.57
5,Educational,280099,17.36
6,Driver_App,173288,10.74


#### LTR-Segment

In [27]:
ltr_segment_city = df_customer_data\
                        .groupby(['ltr_segment'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
ltr_segment_city['% city threshold'] =  (ltr_segment_city['customers']*100.00/ltr_segment_city.customers.sum()).round(2)
ltr_segment_city

,ltr_segment,customers,% city threshold
0,HH,253601,15.72
1,LTR 0,29045,1.80
2,PHH,1329351,82.41
3,UNKNOWN,1089,0.07


In [28]:
df1 = df_customer_data_explode[~df_customer_data_explode['ltr_segment'].isin(['UNKNOWN'])]\
.groupby(['categories_l2_y', 'ltr_segment'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'customers'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)
df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2 = pd.merge(df2,ltr_segment_city[['ltr_segment','customers']], how= 'left', on='ltr_segment')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='ltr_segment' , columns ='categories_l2', values =['customers'])

customers                                                 \
categories_l2 Driver_App Educational Finance_Investment  Gaming  Office   
ltr_segment                                                               
HH                 23851       37421              45716   42944  108194   
LTR 0               2686        3855               4335    5099   10761   
PHH               146697      238592             377865  235299  748065   

                                    
categories_l2     Rest Ride haling  
ltr_segment                         
HH              253381      125931  
LTR 0            29018       13190  
PHH            1328701      934091

In [29]:
# print('% Distribution of customers across individual categories.')
# df2.pivot(index ='ltr_segment' , columns ='categories_l2', values =['customer %'])

In [30]:
print('% Distribution of customers across individual Segment.')
df2.pivot(index ='ltr_segment' , columns ='categories_l2', values =['customers %'])

% Distribution of customers across individual Segment.


customers %                                                      \
categories_l2  Driver_App Educational Finance_Investment Gaming Office   Rest   
ltr_segment                                                                     
HH                   9.40       14.76              18.03  16.93  42.66  99.91   
LTR 0                9.25       13.27              14.93  17.56  37.05  99.91   
PHH                 11.04       17.95              28.42  17.70  56.27  99.95   

                           
categories_l2 Ride haling  
ltr_segment                
HH                  49.66  
LTR 0               45.41  
PHH                 70.27

#### Other testing

In [31]:
df_test = df_customer_app_cat_mapping[df_customer_app_cat_mapping['categories_l2'].isin(['Office', 
                                                                                         'Educational',
                                                                                         'Ride haling'
                                                                                        ])]
df_test = pd.merge(df_test,df_bangalore_active_customer[['customer_id', 'ltr_segment']], how='inner', on='customer_id')
df_office = df_test[df_test['categories_l2'].isin(['Office'])]
df_education = df_test[df_test['categories_l2'].isin(['Educational'])]
df_ride_hailing = df_test[df_test['categories_l2'].isin(['Ride haling'])]

##### Explode - Ride Hailing

In [32]:
rh_ltr_segment_city = df_ride_hailing\
                        .groupby(['ltr_segment'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
rh_ltr_segment_city['% city threshold'] =  (rh_ltr_segment_city['customers']*100.00/rh_ltr_segment_city.customers.sum()).round(2)
rh_ltr_segment_city

,ltr_segment,customers,% city threshold
0,HH,125931,11.72
1,LTR 0,13190,1.23
2,PHH,934091,86.97
3,UNKNOWN,856,0.08


In [33]:
df1 = df_ride_hailing[~df_ride_hailing['ltr_segment'].isin(['UNKNOWN'])]\
.groupby(['app_name', 'ltr_segment'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['app_name', 'customers'],ascending=[False,True])\
.reset_index()
ride_hailing_total_customers = df_ride_hailing.customer_id.nunique()
df2 = pd.merge(df1,rh_ltr_segment_city[['ltr_segment','customers']], how= 'left', on='ltr_segment')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/ride_hailing_total_customers).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customers'])

customers                                                        \
app_name     blusmart driveu driver app in drive jugnoo namma yatri     ola   
ltr_segment                                                                   
HH                237               298     2175     44        1351   94239   
LTR 0              26                18      272      5         147    9621   
PHH              2484              4082    17672    464       15545  725728   

                                          
app_name    quick ride quickride    uber  
ltr_segment                               
HH                 453       196   81867  
LTR 0               49        50    8497  
PHH               6282       736  684770

In [34]:
# print('% Distribution of customers across individual apps.')
# df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customer %'])

In [35]:
print('% Distribution of customers across individual segment.')
df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customers %'])

% Distribution of customers across individual segment.


customers %                                                       \
app_name       blusmart driveu driver app in drive jugnoo namma yatri    ola   
ltr_segment                                                                    
HH                 0.19              0.24     1.73   0.03        1.07  74.83   
LTR 0              0.20              0.14     2.06   0.04        1.11  72.94   
PHH                0.27              0.44     1.89   0.05        1.66  77.69   

                                         
app_name    quick ride quickride   uber  
ltr_segment                              
HH                0.36      0.16  65.01  
LTR 0             0.37      0.38  64.42  
PHH               0.67      0.08  73.31

##### Explode - Office

In [36]:
df_office.head(2)

,customer_id,app_list_set,app_name,categories_l2,categories_l1,ltr_segment
1,644accde8fac8a78c2c1f759,outlook,outlook,Office,Office,PHH
2,644accde8fac8a78c2c1f759,microsoft teams,microsoft teams,Office,Office,PHH


In [37]:
def assign_office_value(row):
    if row['app_name'] in ['asana','cisco','confluence','github','google analytics',
                           'intune  company portal','jira','miro','slack','trello','zoho mail','zoho meeting']:
        return 'code_office_app'
    else:
        return 'secondary_office_app'

df_office['app_name_tag'] = df_office.apply(assign_office_value, axis=1)

In [38]:
df_office[df_office['app_name_tag'] == 'code_office_app'].app_name.unique()

array(['intune  company portal', 'slack', 'github', 'zoho meeting',
       'google analytics', 'jira', 'asana', 'zoho mail', 'cisco',
       'trello', 'miro', 'confluence'], dtype=object)

In [39]:
office_ltr_segment_city = df_office\
                        .groupby(['ltr_segment'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
office_ltr_segment_city['% city threshold'] =  (office_ltr_segment_city['customers']*100.00/office_ltr_segment_city.customers.sum()).round(2)
office_ltr_segment_city

,ltr_segment,customers,% city threshold
0,HH,108194,12.47
1,LTR 0,10761,1.24
2,PHH,748065,86.20
3,UNKNOWN,787,0.09


In [40]:
df1 = df_office[~df_office['ltr_segment'].isin(['UNKNOWN'])]\
.groupby(['app_name', 'ltr_segment'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['app_name', 'customers'],ascending=[False,True])\
.reset_index()
office_total_customers = df_office.customer_id.nunique()
df2 = pd.merge(df1,office_ltr_segment_city[['ltr_segment','customers']], how= 'left', on='ltr_segment')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/office_total_customers).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customers'])

customers                                                      \
app_name        asana  cisco confluence  dropbox  github google analytics   
ltr_segment                                                                 
HH               51.0   93.0       64.0   1328.0   692.0             98.0   
LTR 0             7.0    7.0        5.0    113.0    66.0             11.0   
PHH             790.0  853.0      721.0  11837.0  5896.0           1098.0   

                                                                               \
app_name    google authenticator intune  company portal    jira microsoft 365   
ltr_segment                                                                     
HH                        5806.0                11915.0   261.0       36525.0   
LTR 0                      487.0                  976.0    14.0        3703.0   
PHH                      57308.0               123541.0  3755.0      274128.0   

                                                                       \
app_name    microsoft teams   miro ms authenticator nishtha  onedrive   
ltr_segment                                                             
HH                  27061.0   30.0          15957.0     1.0   49272.0   
LTR 0                2313.0    1.0           1374.0     NaN    5370.0   
PHH                246357.0  332.0         160969.0     2.0  320662.0   

                                                                               \
app_name      outlook pocket    slack  trello    webex zoho mail zoho meeting   
ltr_segment                                                                     
HH            37103.0   91.0   1951.0   101.0   5849.0     359.0        263.0   
LTR 0          3665.0   11.0    178.0    12.0    564.0      32.0         16.0   
PHH          297231.0  949.0  22779.0  1285.0  48255.0    4244.0       2156.0   

                       
app_name         zoom  
ltr_segment            
HH            54604.0  
LTR 0          5278.0  
PHH          386248.0

In [41]:
# print('% Distribution of customers across individual categories.')
df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customers %'])

customers %                                                   \
app_name          asana cisco confluence dropbox github google analytics   
ltr_segment                                                                
HH                 0.05  0.09       0.06    1.23   0.64             0.09   
LTR 0              0.07  0.07       0.05    1.05   0.61             0.10   
PHH                0.11  0.11       0.10    1.58   0.79             0.15   

                                                                             \
app_name    google authenticator intune  company portal  jira microsoft 365   
ltr_segment                                                                   
HH                          5.37                  11.01  0.24         33.76   
LTR 0                       4.53                   9.07  0.13         34.41   
PHH                         7.66                  16.51  0.50         36.64   

                                                                             \
app_name    microsoft teams  miro ms authenticator nishtha onedrive outlook   
ltr_segment                                                                   
HH                    25.01  0.03            14.75     0.0    45.54   34.29   
LTR 0                 21.49  0.01            12.77     NaN    49.90   34.06   
PHH                   32.93  0.04            21.52     0.0    42.87   39.73   

                                                                     
app_name    pocket slack trello webex zoho mail zoho meeting   zoom  
ltr_segment                                                          
HH            0.08  1.80   0.09  5.41      0.33         0.24  50.47  
LTR 0         0.10  1.65   0.11  5.24      0.30         0.15  49.05  
PHH           0.13  3.05   0.17  6.45      0.57         0.29  51.63

In [42]:
print('% Distribution of customers across individual app.')
df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customers %'])

% Distribution of customers across individual app.


customers %                                                   \
app_name          asana cisco confluence dropbox github google analytics   
ltr_segment                                                                
HH                 0.05  0.09       0.06    1.23   0.64             0.09   
LTR 0              0.07  0.07       0.05    1.05   0.61             0.10   
PHH                0.11  0.11       0.10    1.58   0.79             0.15   

                                                                             \
app_name    google authenticator intune  company portal  jira microsoft 365   
ltr_segment                                                                   
HH                          5.37                  11.01  0.24         33.76   
LTR 0                       4.53                   9.07  0.13         34.41   
PHH                         7.66                  16.51  0.50         36.64   

                                                                             \
app_name    microsoft teams  miro ms authenticator nishtha onedrive outlook   
ltr_segment                                                                   
HH                    25.01  0.03            14.75     0.0    45.54   34.29   
LTR 0                 21.49  0.01            12.77     NaN    49.90   34.06   
PHH                   32.93  0.04            21.52     0.0    42.87   39.73   

                                                                     
app_name    pocket slack trello webex zoho mail zoho meeting   zoom  
ltr_segment                                                          
HH            0.08  1.80   0.09  5.41      0.33         0.24  50.47  
LTR 0         0.10  1.65   0.11  5.24      0.30         0.15  49.05  
PHH           0.13  3.05   0.17  6.45      0.57         0.29  51.63

In [43]:
df1 = df_office[~df_office['ltr_segment'].isin(['UNKNOWN'])]\
.groupby(['app_name_tag', 'ltr_segment'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['app_name_tag', 'customers'],ascending=[False,True])\
.reset_index()
office_total_customers = df_office.customer_id.nunique()
df2 = pd.merge(df1,office_ltr_segment_city[['ltr_segment','customers']], how= 'left', on='ltr_segment')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/office_total_customers).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='ltr_segment' , columns ='app_name_tag', values =['customers'])

customers                     
app_name_tag code_office_app secondary_office_app
ltr_segment                                      
HH                     14710               107418
LTR 0                   1233                10691
PHH                   153561               740825

In [44]:
#### print('% Distribution of customers across individual app.')
df2.pivot(index ='ltr_segment' , columns ='app_name_tag', values =['customers %'])

customers %                     
app_name_tag code_office_app secondary_office_app
ltr_segment                                      
HH                     13.60                99.28
LTR 0                  11.46                99.35
PHH                    20.53                99.03

##### Explode - Education

In [45]:
df_education.head(2)

,customer_id,app_list_set,app_name,categories_l2,categories_l1,ltr_segment
15,5e01c812277dbb0268bb9c11,unacademy,unacademy,Educational,Educational,PHH
24,634e4d400be20ada4595f5cd,brainly,brainly,Educational,Educational,PHH


In [46]:
def assign_education_value(row):
    if row['app_name'] in ['aakash','byju','chegg study','diksha','e-pg pathshala',
                           'google classroom', 'microsoft math solver','moodle','mycbseguide', 
                           'physics wallah','simplilearn','vedantu','vidyakul'
                          ]:
#         brainly duolingo
        return 'code_education_app'
    else:
        return 'secondary_education_app'

df_education['app_name_tag'] = df_education.apply(assign_education_value, axis=1)
df_education.head(2)

,customer_id,app_list_set,app_name,categories_l2,categories_l1,ltr_segment,app_name_tag
15,5e01c812277dbb0268bb9c11,unacademy,unacademy,Educational,Educational,PHH,secondary_education_app
24,634e4d400be20ada4595f5cd,brainly,brainly,Educational,Educational,PHH,secondary_education_app


In [47]:
df_education[df_education['app_name_tag'] == 'code_education_app'].app_name.unique()

array(['byju', 'google classroom', 'physics wallah', 'simplilearn',
       'diksha', 'microsoft math solver', 'e-pg pathshala', 'vedantu',
       'aakash', 'moodle', 'mycbseguide', 'chegg study', 'vidyakul'],
      dtype=object)

In [48]:
edu_ltr_segment_city = df_education\
                        .groupby(['ltr_segment'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
edu_ltr_segment_city['% city threshold'] =  (edu_ltr_segment_city['customers']*100.00/edu_ltr_segment_city.customers.sum()).round(2)
edu_ltr_segment_city

,ltr_segment,customers,% city threshold
0,HH,37421,13.36
1,LTR 0,3855,1.38
2,PHH,238592,85.18
3,UNKNOWN,231,0.08


In [49]:
df1 = df_education[~df_education['ltr_segment'].isin(['UNKNOWN'])]\
.groupby(['app_name', 'ltr_segment'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['app_name', 'customers'],ascending=[False,True])\
.reset_index()
edu_total_customers = df_education.customer_id.nunique()
df2 = pd.merge(df1,edu_ltr_segment_city[['ltr_segment','customers']], how= 'left', on='ltr_segment')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/edu_total_customers).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customers'])

customers                                                         \
app_name       aakash  brainly     byju caclubindia chegg study colegeduniya   
ltr_segment                                                                    
HH              215.0   6309.0   6925.0        15.0       131.0         22.0   
LTR 0            29.0    581.0    799.0         NaN        11.0          4.0   
PHH            1160.0  39665.0  34330.0       158.0       997.0        165.0   

                                                                         \
app_name    collegedekho course hero coursera  diksha doubtnut duolingo   
ltr_segment                                                               
HH                   2.0        14.0   1527.0  1266.0   1420.0   6826.0   
LTR 0                NaN         1.0    158.0   133.0    181.0    719.0   
PHH                 15.0        86.0  14017.0  5742.0   6585.0  45551.0   

                                                                             \
app_name    e-pg pathshala     edx embibe fiitjee geeks for geeks goodreads   
ltr_segment                                                                   
HH                   145.0   226.0  135.0     1.0           474.0     220.0   
LTR 0                 17.0    30.0   11.0     1.0            54.0      25.0   
PHH                  978.0  2196.0  802.0     9.0          3802.0    2463.0   

                                                                   \
app_name    google classroom happify ignou e-content khan academy   
ltr_segment                                                         
HH                    7967.0     6.0           103.0        182.0   
LTR 0                  774.0     NaN             8.0         28.0   
PHH                  49476.0    31.0           856.0       1590.0   

                                                                     \
app_name    microsoft math solver  moodle my study life mycbseguide   
ltr_segment                                                           
HH                          105.0   452.0          10.0       105.0   
LTR 0                        16.0    45.0           NaN         9.0   
PHH                         766.0  2831.0          74.0       596.0   

                                                                       \
app_name    ncert books photomath physics wallah pocket shiksha mitra   
ltr_segment                                                             
HH                229.0     395.0         1787.0   91.0           NaN   
LTR 0              26.0      45.0          226.0   11.0           NaN   
PHH              1349.0    2168.0         9742.0  949.0          10.0   

                                                                            \
app_name    shiksha.com simplilearn stack overflow   swayam toppr    udemy   
ltr_segment                                                                  
HH                 54.0       756.0            4.0   1985.0  13.0   5210.0   
LTR 0               9.0        64.0            NaN    188.0   2.0    426.0   
PHH               352.0      5458.0           38.0  10111.0  69.0  54336.0   

                                                    
app_name    unacademy vedantu vidyakul whitehat jr  
ltr_segment                                         
HH             3650.0   498.0      8.0       149.0  
LTR 0           400.0    55.0      5.0        18.0  
PHH           20827.0  2461.0     22.0       965.0

In [50]:
# print('% Distribution of customers across individual apps.')
# df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customer %'])

In [51]:
print('% Distribution of customers across individual segment.')
df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customers %'])

% Distribution of customers across individual segment.


customers %                                                      \
app_name         aakash brainly   byju caclubindia chegg study colegeduniya   
ltr_segment                                                                   
HH                 0.57   16.86  18.51        0.04        0.35         0.06   
LTR 0              0.75   15.07  20.73         NaN        0.29         0.10   
PHH                0.49   16.62  14.39        0.07        0.42         0.07   

                                                                        \
app_name    collegedekho course hero coursera diksha doubtnut duolingo   
ltr_segment                                                              
HH                  0.01        0.04     4.08   3.38     3.79    18.24   
LTR 0                NaN        0.03     4.10   3.45     4.70    18.65   
PHH                 0.01        0.04     5.87   2.41     2.76    19.09   

                                                                           \
app_name    e-pg pathshala   edx embibe fiitjee geeks for geeks goodreads   
ltr_segment                                                                 
HH                    0.39  0.60   0.36    0.00            1.27      0.59   
LTR 0                 0.44  0.78   0.29    0.03            1.40      0.65   
PHH                   0.41  0.92   0.34    0.00            1.59      1.03   

                                                                   \
app_name    google classroom happify ignou e-content khan academy   
ltr_segment                                                         
HH                     21.29    0.02            0.28         0.49   
LTR 0                  20.08     NaN            0.21         0.73   
PHH                    20.74    0.01            0.36         0.67   

                                                                    \
app_name    microsoft math solver moodle my study life mycbseguide   
ltr_segment                                                          
HH                           0.28   1.21          0.03        0.28   
LTR 0                        0.42   1.17           NaN        0.23   
PHH                          0.32   1.19          0.03        0.25   

                                                                       \
app_name    ncert books photomath physics wallah pocket shiksha mitra   
ltr_segment                                                             
HH                 0.61      1.06           4.78   0.24           NaN   
LTR 0              0.67      1.17           5.86   0.29           NaN   
PHH                0.57      0.91           4.08   0.40           0.0   

                                                                        \
app_name    shiksha.com simplilearn stack overflow swayam toppr  udemy   
ltr_segment                                                              
HH                 0.14        2.02           0.01   5.30  0.03  13.92   
LTR 0              0.23        1.66            NaN   4.88  0.05  11.05   
PHH                0.15        2.29           0.02   4.24  0.03  22.77   

                                                    
app_name    unacademy vedantu vidyakul whitehat jr  
ltr_segment                                         
HH               9.75    1.33     0.02        0.40  
LTR 0           10.38    1.43     0.13        0.47  
PHH              8.73    1.03     0.01        0.40

In [52]:
df1 = df_education[~df_education['ltr_segment'].isin(['UNKNOWN'])]\
.groupby(['app_name_tag', 'ltr_segment'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['app_name_tag', 'customers'],ascending=[False,True])\
.reset_index()
edu_total_customers = df_education.customer_id.nunique()
df2 = pd.merge(df1,edu_ltr_segment_city[['ltr_segment','customers']], how= 'left', on='ltr_segment')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/edu_total_customers).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='ltr_segment' , columns ='app_name_tag', values =['customers'])

customers                        
app_name_tag code_education_app secondary_education_app
ltr_segment                                            
HH                        18852                   23827
LTR 0                      2004                    2400
PHH                      105581                  166461

In [53]:
print('% Distribution of customers across individual segment.')
df2.pivot(index ='ltr_segment' , columns ='app_name_tag', values =['customers %'])

% Distribution of customers across individual segment.


customers %                        
app_name_tag code_education_app secondary_education_app
ltr_segment                                            
HH                        50.38                   63.67
LTR 0                     51.98                   62.26
PHH                       44.25                   69.77

### Other

#### Service Affinity

In [54]:
service_affinity_city = df_customer_data\
                        .groupby(['service_affinity'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
service_affinity_city['% city threshold']=(service_affinity_city['customers']*100.00/service_affinity_city.customers.sum()).round(2)
service_affinity_city.sort_values(by=['service_affinity'],ascending=[True])

,service_affinity,customers,% city threshold
0,AUTO_CAB,33526,2.08
1,LINK_AUTO,177730,11.02
2,LINK_CAB,11640,0.72
3,NO_AFFINITY,147463,9.14
4,ONLY_AUTO,557016,34.53
5,ONLY_CAB,47125,2.92
6,ONLY_LINK,565077,35.03
7,UNKNOWN,73509,4.56


In [55]:
df1 = df_customer_data_explode[~df_customer_data_explode['service_affinity'].isin(['UNKNOWN'])]\
.groupby(['categories_l2_y', 'service_affinity'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'customers'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)
df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2 = pd.merge(df2,service_affinity_city[['service_affinity','customers']], how= 'left', on='service_affinity')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='service_affinity' , columns ='categories_l2', values =['customers'])

customers                                                 \
categories_l2    Driver_App Educational Finance_Investment  Gaming  Office   
service_affinity                                                             
AUTO_CAB               2106        6811              10285    5148   21702   
LINK_AUTO             22711       29217              46949   33340   90162   
LINK_CAB               1598        1925               4122    1949    6716   
NO_AFFINITY           17303       26353              48501   26273   87506   
ONLY_AUTO             41055      102585             124877   95563  307018   
ONLY_CAB               3676        9155              16321    6961   29530   
ONLY_LINK             76824       91379             158340  100682  286946   

                                      
categories_l2       Rest Ride haling  
service_affinity                      
AUTO_CAB           33503       26603  
LINK_AUTO         177652      116212  
LINK_CAB           11630        8083  
NO_AFFINITY       147409      112421  
ONLY_AUTO         556610      390233  
ONLY_CAB           47094       35964  
ONLY_LINK         564815      340808

In [56]:
print('% Distribution of customers across individual Segment.')
df2.pivot(index ='service_affinity' , columns ='categories_l2', values =['customers %'])

% Distribution of customers across individual Segment.


customers %                                               \
categories_l2     Driver_App Educational Finance_Investment Gaming Office   
service_affinity                                                            
AUTO_CAB                6.28       20.32              30.68  15.36  64.73   
LINK_AUTO              12.78       16.44              26.42  18.76  50.73   
LINK_CAB               13.73       16.54              35.41  16.74  57.70   
NO_AFFINITY            11.73       17.87              32.89  17.82  59.34   
ONLY_AUTO               7.37       18.42              22.42  17.16  55.12   
ONLY_CAB                7.80       19.43              34.63  14.77  62.66   
ONLY_LINK              13.60       16.17              28.02  17.82  50.78   

                                     
categories_l2      Rest Ride haling  
service_affinity                     
AUTO_CAB          99.93       79.35  
LINK_AUTO         99.96       65.39  
LINK_CAB          99.91       69.44  
NO_AFFINITY       99.96       76.24  
ONLY_AUTO         99.93       70.06  
ONLY_CAB          99.93       76.32  
ONLY_LINK         99.95       60.31

Insights 

- Whenever an app belongs to the app-categories listed below, customers are more likely to be LINK Customers.
    - Driver_App, Finance_Investment
- Whenever an app belongs to the app-categories listed below, customers are more likely to be AUTO Customers.
    - Educational, Office, Ride haling

#### Distance Preference

In [57]:
distance_affinity_city = df_customer_data\
                        .groupby(['distance_preference'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
distance_affinity_city['% city threshold']=(distance_affinity_city['customers']*100.00/distance_affinity_city.customers.sum()).round(2)
distance_affinity_city.sort_values(by=['distance_preference'],ascending=[True])

,distance_preference,customers,% city threshold
0,LONG_DISTANCE,414674,25.71
1,MEDIUM_DISTANCE,369854,22.93
2,SHORT_DISTANCE,364464,22.59
3,UNKNOWN,464094,28.77


In [58]:
df1 = df_customer_data_explode\
.groupby(['categories_l2_y', 'distance_preference'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'distance_preference'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)

df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2.pivot(index ='distance_preference' , columns ='categories_l2', values =['customer %'])

customer %                                               \
categories_l2       Driver_App Educational Finance_Investment Gaming Office   
distance_preference                                                           
LONG_DISTANCE            27.72       23.35              26.49  25.57  24.47   
MEDIUM_DISTANCE          21.54       22.70              22.19  22.07  22.76   
SHORT_DISTANCE           18.49       25.14              22.00  22.25  23.63   
UNKNOWN                  32.25       28.81              29.33  30.12  29.13   

                                        
categories_l2         Rest Ride haling  
distance_preference                     
LONG_DISTANCE        25.71       25.91  
MEDIUM_DISTANCE      22.93       22.59  
SHORT_DISTANCE       22.59       22.47  
UNKNOWN              28.77       29.03

Insights 

- Whenever an app belongs to the app-categories listed below, customers are more likely to do LONG Distance rides.
    - Driver_App, Gaming
   
- Whenever an app belongs to the app-categories listed below, customers are more likely to do MEDIUM Distance rides.
    - Finance_Investment
   
- Whenever an app belongs to the app-categories listed below, customers are more likely to do SHORT Distance rides.
    - Educational, Office

#### Gender

In [59]:
gender_city = df_customer_data\
                        .groupby(['gender'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
gender_city['% city threshold']=(gender_city['customers']*100.00/gender_city.customers.sum()).round(2)
gender_city.sort_values(by=['gender'],ascending=[True])

,gender,customers,% city threshold
0,FEMALE,455607,28.24
1,MALE,1105656,68.54
2,OTHERS,6505,0.40
3,UNKNOWN,45318,2.81


In [60]:
df1 = df_customer_data_explode\
.groupby(['categories_l2_y', 'gender'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'gender'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)

df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2.pivot(index ='gender' , columns ='categories_l2', values =['customer %'])

customer %                                                      \
categories_l2 Driver_App Educational Finance_Investment Gaming Office   Rest   
gender                                                                         
FEMALE              8.12       34.93              21.94  31.95  32.24  28.24   
MALE               90.30       61.67              75.55  64.74  64.65  68.54   
OTHERS              0.24        0.30               0.15   0.41   0.29   0.40   
UNKNOWN             1.34        3.09               2.37   2.90   2.83   2.81   

                           
categories_l2 Ride haling  
gender                     
FEMALE              31.32  
MALE                65.56  
OTHERS               0.33  
UNKNOWN              2.79

Insights 

- Whenever an app belongs to the app-categories listed below, customers are more likely to be MALE.
    - Driver_App, Finance_Investment
- Whenever an app belongs to the app-categories listed below, customers are more likely to be FEMALE.
    - Educational

#### Income Segment

In [61]:
income_segment_city = df_customer_data\
                        .groupby(['income_segment'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
income_segment_city['% city threshold']=(income_segment_city['customers']*100.00/income_segment_city.customers.sum()).round(2)
income_segment_city.sort_values(by=['income_segment'],ascending=[True])

,income_segment,customers,% city threshold
0,HIGH_INCOME,624839,38.74
1,LOW_INCOME,110374,6.84
2,MEDIUM_INCOME,630745,39.10
3,UNKNOWN,247128,15.32


In [62]:
df1 = df_customer_data_explode\
.groupby(['categories_l2_y', 'income_segment'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'income_segment'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)

df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2.pivot(index ='income_segment' , columns ='categories_l2', values =['customer %'])

customer %                                                      \
categories_l2  Driver_App Educational Finance_Investment Gaming Office   Rest   
income_segment                                                                  
HIGH_INCOME         41.42       46.81              50.30  44.41  44.43  38.75   
LOW_INCOME           4.76        3.56               2.95   4.39   4.16   6.83   
MEDIUM_INCOME       39.25       38.23              32.30  37.76  38.44  39.10   
UNKNOWN             14.57       11.41              14.45  13.44  12.97  15.32   

                            
categories_l2  Ride haling  
income_segment              
HIGH_INCOME          43.96  
LOW_INCOME            4.69  
MEDIUM_INCOME        37.50  
UNKNOWN              13.85

Insights 

- Whenever an app belongs to the app-categories listed below, customers are more likely to be HIGH_INCOME.
    - Office, Educational, Finance_Investment, Gaming, Ride haling
- Whenever an app belongs to the app-categories listed below, customers are more likely to be MEDIUM_INCOME.
    - Driver_App

#### Price Sensitivity

In [63]:
ps_city = df_customer_data\
                        .groupby(['price_sensitivity'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
ps_city['% city threshold']=(ps_city['customers']*100.00/ps_city.customers.sum()).round(2)
ps_city.sort_values(by=['price_sensitivity'],ascending=[True])

,price_sensitivity,customers,% city threshold
0,NPS,563952,34.96
1,PS,430741,26.70
2,UNKNOWN,618393,38.34


In [64]:
df1 = df_customer_data_explode[~df_customer_data_explode['price_sensitivity'].isin(['UNKNOWN'])]\
.groupby(['categories_l2_y', 'price_sensitivity'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'customers'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)
df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2 = pd.merge(df2,ps_city[['price_sensitivity','customers']], how= 'left', on='price_sensitivity')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='price_sensitivity' , columns ='categories_l2', values =['customers %','customers'])

customers %                                               \
categories_l2      Driver_App Educational Finance_Investment Gaming Office   
price_sensitivity                                                            
NPS                      9.77       18.26              27.18  17.78  56.46   
PS                      10.96       17.23              25.80  17.80  54.16   

                                      customers              \
categories_l2       Rest Ride haling Driver_App Educational   
price_sensitivity                                             
NPS                99.95       71.32    55084.0    102994.0   
PS                 99.94       70.43    47212.0     74223.0   

                                                                                
categories_l2     Finance_Investment    Gaming    Office      Rest Ride haling  
price_sensitivity                                                               
NPS                         153279.0  100262.0  318411.0  563687.0    402198.0  
PS                          111137.0   76675.0  233308.0  430501.0    303389.0

Insights 

- Whenever an app belongs to the app-categories listed below, customers are more likely to be Price Sensitivity.
    - Driver_App
- Whenever an app belongs to the app-categories listed below, customers are less likely to be Price Sensitivity. (Less confidence - Uninterpretable/Hard to interpretable using Price Sensitivity)
    - Office, Educational, Finance_Investment, Ride haling, Gaming

#### RPC

In [65]:
rpc_bucket_city = df_customer_data\
                        .groupby(['rpc_bucket'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
rpc_bucket_city['% city threshold']=(rpc_bucket_city['customers']*100.00/rpc_bucket_city.customers.sum()).round(2)
rpc_bucket_city.sort_values(by=['rpc_bucket'],ascending=[True])

,rpc_bucket,customers,% city threshold
0,a. ZERO,223776,13.87
1,b. LOW,639198,39.63
2,c. MEDIUM,450821,27.95
3,d. HIGH,299204,18.55


In [66]:
df1 = df_customer_data_explode[~df_customer_data_explode['rpc_bucket'].isin(['UNKNOWN'])]\
.groupby(['categories_l2_y', 'rpc_bucket'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'customers'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)
df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2 = pd.merge(df2,rpc_bucket_city[['rpc_bucket','customers']], how= 'left', on='rpc_bucket')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='rpc_bucket' , columns ='categories_l2', values =['customers %','customers'])

customers %                                                      \
categories_l2  Driver_App Educational Finance_Investment Gaming Office   Rest   
rpc_bucket                                                                      
a. ZERO             12.37       17.44              25.18  18.73  51.37  99.95   
b. LOW              11.62       17.07              26.44  17.77  52.50  99.94   
c. MEDIUM           10.30       17.39              26.96  17.28  54.77  99.94   
d. HIGH              8.31       17.90              27.20  16.72  56.93  99.94   

                           customers                                           \
categories_l2 Ride haling Driver_App Educational Finance_Investment    Gaming   
rpc_bucket                                                                      
a. ZERO             67.18    27673.0     39026.0            56343.0   41918.0   
b. LOW              64.04    74297.0    109117.0           168998.0  113590.0   
c. MEDIUM           66.60    46456.0     78387.0           121529.0   77917.0   
d. HIGH             71.54    24853.0     53556.0            81374.0   50012.0   

                                               
categories_l2    Office      Rest Ride haling  
rpc_bucket                                     
a. ZERO        114943.0  223659.0    150333.0  
b. LOW         335558.0  638836.0    409374.0  
c. MEDIUM      246916.0  450571.0    300256.0  
d. HIGH        170348.0  299036.0    214053.0

Insights 

- Whenever an app belongs to the app-categories listed below, customers having Less RPC
    - Driver_App
- Whenever an app belongs to the app-categories listed below, customers having Good RPC
    - Educational, Finance_Investment, Office, Ride haling

#### App Count Bucket

In [67]:
app_count_bucket_city = df_customer_data\
                        .groupby(['app_count_bucket'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
app_count_bucket_city['% city threshold']=(app_count_bucket_city['customers']*100.00/app_count_bucket_city.customers.sum()).round(2)
app_count_bucket_city.sort_values(by=['app_count_bucket'],ascending=[True])

,app_count_bucket,customers,% city threshold
0,1-5,1402987,86.98
1,11-15,40102,2.49
2,6-10,152148,9.43
3,Above 15,17849,1.11


In [68]:
df1 = df_customer_data_explode\
.groupby(['categories_l2_y', 'app_count_bucket'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'app_count_bucket'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)

df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2.pivot(index ='app_count_bucket' , columns ='categories_l2', values =['customer %'])

customer %                                               \
categories_l2    Driver_App Educational Finance_Investment Gaming Office   
app_count_bucket                                                           
1-5                   90.33       86.57              86.70  87.65  86.17   
11-15                  1.64        2.59               2.58   2.29   2.66   
6-10                   7.29        9.77               9.59   9.10  10.02   
Above 15               0.74        1.07               1.12   0.96   1.15   

                                     
categories_l2      Rest Ride haling  
app_count_bucket                     
1-5               86.98       85.84  
11-15              2.49        2.76  
6-10               9.43       10.16  
Above 15           1.11        1.24

In [69]:
df_customer_data.app_count.describe()

count    1.613086e+06
mean     1.756618e+01
std      8.856690e+00
min      1.000000e+00
25%      1.100000e+01
50%      1.600000e+01
75%      2.300000e+01
max      9.400000e+01
Name: app_count, dtype: float64

#### Category Count Bucket

In [70]:
cat_count_bucket_city = df_customer_data\
                        .groupby(['category_count_bucket'])\
                        .agg(customers = ('customer_id','nunique'),)\
                        .reset_index()
cat_count_bucket_city['% city threshold']=(cat_count_bucket_city['customers']*100.00/cat_count_bucket_city.customers.sum()).round(2)
cat_count_bucket_city.sort_values(by=['category_count_bucket'],ascending=[True])

,category_count_bucket,customers,% city threshold
0,1,220094,13.64
1,2-3,871988,54.06
2,Above 3,521004,32.30


In [71]:
df1 = df_customer_data_explode\
.groupby(['categories_l2_y', 'category_count_bucket'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'category_count_bucket'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)

df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2.pivot(index ='category_count_bucket' , columns ='categories_l2', values =['customer %'])

customer %                                               \
categories_l2         Driver_App Educational Finance_Investment Gaming Office   
category_count_bucket                                                           
1                           0.02        0.01               0.02   0.01   0.02   
2-3                        45.58       21.50              27.20  37.65  46.18   
Above 3                    54.40       78.50              72.78  62.34  53.80   

                                          
categories_l2           Rest Ride haling  
category_count_bucket                     
1                      13.61        0.03  
2-3                    54.07       53.91  
Above 3                32.32       46.07

In [72]:
df_temp = df_customer_data[df_customer_data['category_count_bucket'].isin(['2-3'])]\
.groupby(['customer_id'])\
.agg({'categories_l2' : set})\
.reset_index()
df_temp[['categories_l2']]

,categories_l2
0,"{['Ride haling', 'Office', 'Rest']}"
1,"{['Ride haling', 'Office', 'Rest']}"
2,"{['Ride haling', 'Office', 'Rest']}"
3,"{['Ride haling', 'Rest']}"
4,"{['Ride haling', 'Office', 'Rest']}"
...,...
871983,"{['Office', 'Rest']}"
871984,"{['Ride haling', 'Rest']}"
871985,"{['Gaming', 'Office', 'Rest']}"
871986,"{['Finance_Investment', 'Office', 'Rest']}"


In [73]:
df_customer_data.category_count.describe()

count    1.613086e+06
mean     2.925554e+00
std      1.235463e+00
min      1.000000e+00
25%      2.000000e+00
50%      3.000000e+00
75%      4.000000e+00
max      7.000000e+00
Name: category_count, dtype: float64

#### Customer Use-Case 

In [74]:
usecase_tag_city = df_customer_data\
                        .groupby(['usecase_tag'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
usecase_tag_city['% city threshold']=(usecase_tag_city['customers']*100.00/usecase_tag_city.customers.sum()).round(2)
usecase_tag_city.sort_values(by=['usecase_tag'],ascending=[True])

,usecase_tag,customers,% city threshold
0,Unknown,231974,14.38
1,educational,64539,4.00
2,food,51902,3.22
3,govt_amenity,26374,1.64
4,health_and_personal,89628,5.56
5,household_needs,94076,5.83
6,leisure,100605,6.24
7,office,125827,7.80
8,place_of_worship,27270,1.69
9,residential,570417,35.36


In [75]:
df1 = df_customer_data_explode\
.groupby(['categories_l2_y', 'usecase_tag'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'usecase_tag'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)

df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2.pivot(index ='usecase_tag' , columns ='categories_l2', values =['customer %'])

customer %                                               \
categories_l2       Driver_App Educational Finance_Investment Gaming Office   
usecase_tag                                                                   
Unknown                  16.51       13.75              13.63  14.58  13.38   
educational               3.73        4.49               3.47   3.83   4.05   
food                      3.28        2.95               2.89   3.12   3.12   
govt_amenity              1.57        1.49               1.53   1.61   1.57   
health_and_personal       5.63        5.11               5.07   5.36   5.39   
household_needs           5.80        5.12               5.04   5.82   5.47   
leisure                   5.87        6.56               6.40   6.21   6.45   
office                    6.45        9.30              10.23   7.81   9.14   
place_of_worship          1.81        1.41               1.33   1.63   1.51   
residential              34.55       35.27              34.85  35.09  35.78   
transit_station          14.80       14.55              15.56  14.93  14.15   

                                        
categories_l2         Rest Ride haling  
usecase_tag                             
Unknown              14.38       13.92  
educational           4.00        4.09  
food                  3.22        3.20  
govt_amenity          1.63        1.63  
health_and_personal   5.56        5.54  
household_needs       5.83        5.72  
leisure               6.24        6.39  
office                7.80        8.36  
place_of_worship      1.69        1.65  
residential          35.36       36.24  
transit_station      14.29       13.26

#### AO-NET Funnel

In [76]:
df_customer_data['city'] = 'Hyderabad, Android'
city_funnel = df_customer_data\
                        .groupby(['city'])\
                        .agg(fe_count = ('fe_count','sum'),
                             rr_count = ('rr_count','sum'),
                             net_count = ('net_count','sum')
                            )\
                        .reset_index()

city_funnel['% fe2rr']=(city_funnel['rr_count']*100.00/city_funnel['fe_count']).round(2)
city_funnel['% g2n']=(city_funnel['net_count']*100.00/city_funnel['rr_count']).round(2)
city_funnel['% fe2net']=(city_funnel['net_count']*100.00/city_funnel['fe_count']).round(2)
city_funnel[['city','% fe2rr','% g2n','% fe2net']].T

,0
city,"Hyderabad, Android"
% fe2rr,45.31
% g2n,63.73
% fe2net,28.88


In [77]:
df1 = df_customer_data_explode\
.groupby(['categories_l2_y'])\
.agg(customers = ('customer_id','nunique'),
     fe_count = ('fe_count','sum'),
     rr_count = ('rr_count','sum'),
     net_count = ('net_count','sum')
    )\
.sort_values(by=['categories_l2_y'],ascending=[False])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)

df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2['% fe2rr']=(df2['rr_count']*100.00/df2['fe_count']).round(2)
df2['% g2n']=(df2['net_count']*100.00/df2['rr_count']).round(2)
df2['% fe2net']=(df2['net_count']*100.00/df2['fe_count']).round(2)
df2[['categories_l2','% fe2rr','% g2n','% fe2net']].T

,0,1,2,3,4,5,6
categories_l2,Rest,Ride haling,Office,Finance_Investment,Gaming,Educational,Driver_App
% fe2rr,45.31,46.57,46.18,45.29,44.98,45.63,44.08
% g2n,63.73,63.59,64.79,65.44,62.57,63.95,62.39
% fe2net,28.88,29.61,29.92,29.64,28.14,29.18,27.5
